In [2]:
!pip install torch torchvision

   ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
    --------------------------------------- 1.8/113.8 MB 14.3 MB/s eta 0:00:08
   --- ------------------------------------ 11.3/113.8 MB 33.6 MB/s eta 0:00:04
   -------- ------------------------------- 24.6/113.8 MB 45.8 MB/s eta 0:00:02
   -------------- ------------------------- 40.4/113.8 MB 54.6 MB/s eta 0:00:02
   --------------------- ------------------ 60.6/113.8 MB 63.3 MB/s eta 0:00:01
   --------------------------- ------------ 78.9/113.8 MB 68.0 MB/s eta 0:00:01
   ---------------------------------- ----- 98.0/113.8 MB 71.9 MB/s eta 0:00:01
   --------------------------------------  113.5/113.8 MB 73.9 MB/s eta 0:00:01
   --------------------------------------- 113.8/113.8 MB 68.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---------------------------------------- 4.3/4.3 MB 86.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/6.3 MB ? eta -:-


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# โค้ดสำหรับติดตั้งโมดูลที่ขาดหายไป (ให้เพิ่มเซลล์ใหม่แล้วรันคำสั่งนี้)
!pip install onnx onnxscript

   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   - -------------------------------------- 0.8/16.4 MB 11.2 MB/s eta 0:00:02
   ------------------------------- -------- 12.8/16.4 MB 44.6 MB/s eta 0:00:01
   ---------------------------------------- 16.4/16.4 MB 46.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/689.1 kB ? eta -:--:--
   ---------------------------------------- 689.1/689.1 kB ? eta 0:00:00

   ---------------------------------------- 0/5 [protobuf]
   ---------------------------------------- 0/5 [protobuf]
   ---------------------------------------- 0/5 [protobuf]
   ---------------------------------------- 0/5 [protobuf]
   ---------------------------------------- 0/5 [protobuf]
   ---------------------------------------- 0/5 [protobuf]
   ---------------------------------------- 0/5 [protobuf]
   ---------------------------------------- 0/5 [protobuf]
   ---------------------------------------- 0/5 [protobuf]
   ---------------


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
# ติดตั้ง ultralytics สำหรับเทรน YOLOv8
!pip install ultralytics

INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 8.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/810.4 kB ? eta -:--:--
   --------------------------------------- 810.4/810.4 kB 36.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/45.7 MB ? eta -:--:--
   ---------- ----------------------------- 12.3/45.7 MB 59.4 MB/s eta 0:00:01
   ----------------------- ---------------- 27.0/45.7 MB 65.9 MB/s eta 0:00:01
   ---------------------------------- ----- 39.8/45.7 MB 65.0 MB/s eta 0:00:01
   ---------------------------------------- 45.7/45.7 MB 60.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   --------------- ------------------------ 14.2/36.5 MB 68.4 MB/s eta 0:00:01
   ------------------------


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import cv2
import numpy as np
import os

def saveimage(image, path_prefix, idx):
    path = f"{path_prefix}_{idx}.jpg"
    cv2.imwrite(path, image)
    return idx + 1

def translation(src, tx_l, tx_r):
    rows, cols = src.shape[:2]
    srcTri = np.float32([[0 + tx_l, 0], [cols - 1 + tx_r, 0], [0 + tx_l, rows - 1]])
    dstTri = np.float32([[0, 0], [cols - 1, 0], [0, rows - 1]])
    warp_mat = cv2.getAffineTransform(srcTri, dstTri)
    return cv2.warpAffine(src, warp_mat, (cols, rows))

def rotation(src, angle):
    rows, cols = src.shape[:2]
    center = (cols / 2.0, rows / 2.0)
    rot_mat = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(src, rot_mat, (cols, rows), borderMode=cv2.BORDER_REPLICATE)

def blurr(image, path_prefix, idx):
    for i in range(3, 10, 4):
        blurred = cv2.GaussianBlur(image, (i, i), 0)
        idx = saveimage(blurred, path_prefix, idx)
    return idx

def noise(image, path_prefix, idx):
    for i in range(1, 4):
        result = cv2.normalize(image, None, 0.0, 1.0, cv2.NORM_MINMAX, dtype=cv2.CV_64F)
        noise_mat = np.zeros_like(result)
        cv2.randn(noise_mat, 0, 0.1 + i / 20.0)
        result = result + noise_mat
        result = cv2.normalize(result, None, 0.0, 1.0, cv2.NORM_MINMAX, dtype=cv2.CV_64F)
        result = np.clip(result * 255, 0, 255).astype(np.uint8)
        
        idx = saveimage(result, path_prefix, idx)
        idx = blurr(result, path_prefix, idx)
    return idx

def generate_all_datasets(input_folder, output_folder):
    # กำหนดรายชื่อภาพที่ต้องการ
    image_list = ['crosswalk', 'road-warning', 'school', 'stop', 'stop-ahead', 'tree-traffic','walk-way','mountain1','mountain2','car-trunk','bush']
    
    os.makedirs(output_folder, exist_ok=True)
    
    for filename in os.listdir(input_folder):
        name_without_ext = os.path.splitext(filename)[0]
        
        # เช็คว่าเป็นภาพในรายการที่ต้องการหรือไม่
        if name_without_ext not in image_list:
            continue
            
        image_path = os.path.join(input_folder, filename)
        src = cv2.imread(image_path, 1)
        
        if src is None:
            continue
            
        # สร้างโฟลเดอร์เก็บผลลัพธ์ตามชื่อภาพ
        save_dir = os.path.join(output_folder, name_without_ext)
        os.makedirs(save_dir, exist_ok=True)
        path_prefix = os.path.join(save_dir, name_without_ext)
        
        idx = 0
        src = cv2.resize(src, (64, 64))
        
        src_tx_left1 = translation(src, 0, -10)
        src_tx_left2 = translation(src, 0, -20)
        src_tx_right1 = translation(src, 10, 0)
        src_tx_right2 = translation(src, 20, 0)
        src_ro_left = rotation(src, -5)
        src_ro_right = rotation(src, 5)
        
        # 1. resize
        idx = saveimage(src, path_prefix, idx)
        idx = noise(src, path_prefix, idx)
        
        # 2. left shift
        idx = saveimage(src_tx_left1, path_prefix, idx)
        idx = noise(src_tx_left1, path_prefix, idx)
        idx = saveimage(src_tx_left2, path_prefix, idx)
        idx = noise(src_tx_left2, path_prefix, idx)
        
        # 3. right shift
        idx = saveimage(src_tx_right1, path_prefix, idx)
        idx = noise(src_tx_right1, path_prefix, idx)
        idx = saveimage(src_tx_right2, path_prefix, idx)
        idx = noise(src_tx_right2, path_prefix, idx)
        
        # 4. rotate left
        idx = saveimage(src_ro_left, path_prefix, idx)
        idx = noise(src_ro_left, path_prefix, idx)
        
        # 5. rotate right
        idx = saveimage(src_ro_right, path_prefix, idx)
        idx = noise(src_ro_right, path_prefix, idx)
        
        print(f"สร้าง Dataset สำหรับ {name_without_ext} เสร็จสิ้น: {idx} รูป")

# นำรูปต้นฉบับไปใส่ในโฟลเดอร์ raw_images แล้วนำเครื่องหมาย # ออกเพื่อสั่งรัน
generate_all_datasets("raw_images", "dataset")

สร้าง Dataset สำหรับ bush เสร็จสิ้น: 70 รูป
สร้าง Dataset สำหรับ car-trunk เสร็จสิ้น: 70 รูป
สร้าง Dataset สำหรับ crosswalk เสร็จสิ้น: 70 รูป
สร้าง Dataset สำหรับ mountain1 เสร็จสิ้น: 70 รูป
สร้าง Dataset สำหรับ mountain2 เสร็จสิ้น: 70 รูป
สร้าง Dataset สำหรับ road-warning เสร็จสิ้น: 70 รูป
สร้าง Dataset สำหรับ stop-ahead เสร็จสิ้น: 70 รูป
สร้าง Dataset สำหรับ stop เสร็จสิ้น: 70 รูป
สร้าง Dataset สำหรับ tree-traffic เสร็จสิ้น: 70 รูป
สร้าง Dataset สำหรับ walk-way เสร็จสิ้น: 70 รูป


In [3]:
import os

# โค้ดสรุปการเตรียมโฟลเดอร์จำแนกคลาส (ทำก่อนขั้นตอนการเทรนโมเดล)
target_path = "dataset"

# 1. จำเป็นต้องสร้างโฟลเดอร์เพื่อระบุคลาสทั้งหมดให้ชัดเจนก่อน
os.makedirs(os.path.join(target_path, 'red_sign'), exist_ok=True)   # สำหรับป้ายหยุด
os.makedirs(os.path.join(target_path, 'yellow_sign'), exist_ok=True)# สำหรับป้ายเตือนต่างๆ
os.makedirs(os.path.join(target_path, 'not_sign'), exist_ok=True)   # สำหรับต้นไม้ รถ ทางม้าลาย

# หลังจากเอารูปไปใส่ตามโฟลเดอร์เหล่านี้เสร็จแล้ว...

In [4]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ย่อขนาดภาพเป็น 64x64 และแปลงเป็น Tensor 
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

dataset_dir = "dataset" 
dataset = datasets.ImageFolder(root=dataset_dir, transform=transform)

# โหลดข้อมูลทีละ 32 รูปแบบสุ่ม
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"คลาสที่พบ: {dataset.classes}")
print(f"จำนวนรูปภาพทั้งหมด: {len(dataset)} รูป")

คลาสที่พบ: ['not_sign', 'red_sign', 'yellow_sign']
จำนวนรูปภาพทั้งหมด: 700 รูป


In [5]:
import torch
import torch.nn as nn

# Cell 3: โครงสร้างโมเดล CNN ที่เน้นการแยกรูปทรง
class TrafficSignCNN(nn.Module):
    def __init__(self, num_classes=3): # ปรับเป็น 3 คลาส (Stop, Warning, Background)
        super(TrafficSignCNN, self).__init__()
        
        # Layer 1: ตรวจจับขอบและเส้น (Low-level features)
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2) # จาก 64x64 เหลือ 32x32
        
        # Layer 2: ตรวจจับรูปทรงและเหลี่ยมมุม (High-level features - สำคัญมาก!)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2) # จาก 32x32 เหลือ 16x16
        
        # Fully Connected Layer: ตัดสินใจว่าเป็นรูปอะไร
        # คำนวณขนาด: 32 channels * 16 pixels * 16 pixels
        self.fc1 = nn.Linear(32 * 16 * 16, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(-1, 32 * 16 * 16) # ปรับขนาดให้แบนเพื่อเข้า Linear
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

# สร้างโมเดล
# หากคุณใช้ dataset.classes จากโฟลเดอร์ มันจะเท่ากับจำนวนโฟลเดอร์ที่คุณปั่นไว้
model = TrafficSignCNN(num_classes=len(dataset.classes))
print(f"สร้างโครงสร้างโมเดลเสร็จสิ้น (รองรับ {len(dataset.classes)} คลาส)")

สร้างโครงสร้างโมเดลเสร็จสิ้น (รองรับ 3 คลาส)


In [6]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("ตั้งค่า Optimizer พร้อมเทรน")

ตั้งค่า Optimizer พร้อมเทรน


In [ ]:
import torch

# Cell 5: กระบวนการเทรน (Training Loop)
epochs = 50

# ตรวจสอบลำดับ Index ก่อนเทรน (สำคัญมากสำหรับการนำไปใช้ใน C++)
print("ลำดับคลาสที่โมเดลกำลังเรียนรู้:")
class_names = dataset.classes
for i, name in enumerate(class_names):
    print(f"Index {i} คือ: {name}")

print("\nกำลังเริ่มต้นกระบวนการเทรน...")
model.train() # ตั้งค่าโมเดลให้อยู่ในโหมดเทรน

for epoch in range(epochs):
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in dataloader:
        optimizer.zero_grad()            
        outputs = model(images)          
        loss = criterion(outputs, labels)
        loss.backward()                  
        optimizer.step()                 
        
        running_loss += loss.item()
        
        # คำนวณความแม่นยำ (Accuracy)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    avg_loss = running_loss / len(dataloader)
    accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f} - Accuracy: {accuracy:.2f}%")

print("เทรนเสร็จสิ้น!")

model_save_path = "D:/taffic_sign/traffic_sign_model_3classes.pth"

# บันทึกเฉพาะน้ำหนักของโมเดล (State Dict)
torch.save(model.state_dict(), model_save_path)

print(f"บันทึกโมเดลเรียบร้อยแล้วที่: {model_save_path}")

# --- ส่วนสำคัญ: ตรวจสอบและจดจำลำดับ Class เพื่อไปใช้ใน C++ ---
print("\n*** กรุณาจดจำลำดับ Index เหล่านี้เพื่อไปเขียนเงื่อนไขใน C++ ***")
for i, name in enumerate(dataset.classes):
    print(f"ผลลัพธ์เลข {i} คือป้าย: {name}")

ลำดับคลาสที่โมเดลกำลังเรียนรู้:
Index 0 คือ: not_sign
Index 1 คือ: red_sign
Index 2 คือ: yellow_sign

กำลังเริ่มต้นกระบวนการเทรน...
Epoch [1/50] - Loss: 0.7520 - Accuracy: 66.57%
Epoch [2/50] - Loss: 0.2527 - Accuracy: 94.14%
Epoch [3/50] - Loss: 0.0513 - Accuracy: 99.71%
Epoch [4/50] - Loss: 0.0145 - Accuracy: 100.00%
Epoch [5/50] - Loss: 0.0060 - Accuracy: 100.00%
Epoch [6/50] - Loss: 0.0033 - Accuracy: 100.00%
Epoch [7/50] - Loss: 0.0024 - Accuracy: 100.00%
Epoch [8/50] - Loss: 0.0016 - Accuracy: 100.00%
Epoch [9/50] - Loss: 0.0013 - Accuracy: 100.00%
Epoch [10/50] - Loss: 0.0009 - Accuracy: 100.00%
Epoch [11/50] - Loss: 0.0007 - Accuracy: 100.00%
Epoch [12/50] - Loss: 0.0005 - Accuracy: 100.00%
Epoch [13/50] - Loss: 0.0004 - Accuracy: 100.00%
Epoch [14/50] - Loss: 0.0003 - Accuracy: 100.00%
Epoch [15/50] - Loss: 0.0003 - Accuracy: 100.00%
Epoch [16/50] - Loss: 0.0002 - Accuracy: 100.00%
Epoch [17/50] - Loss: 0.0002 - Accuracy: 100.00%
Epoch [18/50] - Loss: 0.0002 - Accuracy: 100.00

RuntimeError: [enforce fail at inline_container.cc:741] . open file failed with error code: 5

In [58]:
import torch
from torch.onnx import utils
import onnx

# 1. โหลดโมเดลและน้ำหนัก (ต้องระบุจำนวน Class ให้ตรงกับที่เทรนมา)
# แนะนำให้ใช้ตัวแปร len(dataset.classes) เพื่อความยืดหยุ่น
num_classes = len(dataset.classes) 
model = TrafficSignCNN(num_classes=num_classes) # ใช้ชื่อ Class Model ใหม่ที่เราปรับปรุง

# โหลดไฟล์ .pth ที่เทรนเสร็จแล้ว (อย่าลืมเปลี่ยนชื่อไฟล์ให้ตรงกับที่เซฟไว้)
model.load_state_dict(torch.load("traffic_sign_model_3classes.pth"))
model.eval()

# 2. เตรียม Dummy Input (ขนาด 64x64 ตามที่เรา Resize ไว้)
dummy_input = torch.randn(1, 3, 64, 64)

# 3. ส่งออกเป็น ONNX
utils.export(
    model, 
    dummy_input, 
    "D:/TrafficSound/traffic_sign_model.onnx", 
    export_params=True, 
    opset_version=11,          # Opset 11 มาตรฐานสำหรับ OpenCV
    do_constant_folding=True, 
    input_names=['input'], 
    output_names=['output'],
    dynamic_axes=None,
    keep_initializers_as_inputs=True
)

print(f"ส่งออกโมเดล ({num_classes} คลาส) สำเร็จ!")

# 4. ตรวจสอบความเรียบร้อย
onnx_model = onnx.load("D:/TrafficSound/traffic_sign_model.onnx")
print(f"ยืนยันเวอร์ชัน Opset ในไฟล์: {onnx_model.opset_import[0].version}")

ส่งออกโมเดล (3 คลาส) สำเร็จ!
ยืนยันเวอร์ชัน Opset ในไฟล์: 11


In [ ]:
# import yaml
# import os

# # ใช้ Raw String (r"") เพื่อป้องกันปัญหา Backslash ใน Windows
# project_path = r"D:/TrafficSound/dataset"
# yaml_path = r"D:/TrafficSound/data.yaml"

# data_config = {
#     'path': project_path,
#     'train': 'images/train',
#     'val': 'images/val',
#     'names': {
#         0: 'STOP',
#         1: 'WARNING',
#         2: 'NOT_A_SIGN'
#     }
# }

# # สร้างโฟลเดอร์ปลายทางถ้ายังไม่มี
# os.makedirs(os.path.dirname(yaml_path), exist_ok=True)

# # เขียนไฟล์ .yaml
# with open(yaml_path, 'w') as f:
#     yaml.dump(data_config, f, default_flow_style=False)

# print(f"สร้างไฟล์สำเร็จที่: {yaml_path}")

สร้างไฟล์สำเร็จที่: D:/TrafficSound/data.yaml


In [ ]:
# from ultralytics import YOLO

# # 1. โหลดโมเดลตั้งต้น (YOLOv8 Nano เหมาะที่สุดสำหรับงานนี้)
# model = YOLO("yolov8n.pt") 

# # 2. สั่งเทรน (จะมาแทนที่ Loop for epoch เดิมทั้งหมด)
# # ระบบจะจัดการ Optimizer, Loss และการ Save โมเดลให้เองอัตโนมัติ
# results = model.train(
#     data="D:/TrafficSound/data.yaml", # ต้องสร้างไฟล์นี้เพื่อบอกที่เก็บรูป
#     epochs=100, 
#     imgsz=640, 
#     device="cpu" 
# )

# # 3. ส่งออกโมเดลไปใช้ใน Visual Studio (แทนที่ torch.onnx.export เดิม)
# model.export(format="onnx", opset=11)

Ultralytics 8.4.21  Python-3.12.4 torch-2.10.0+cpu CPU (13th Gen Intel Core i7-13620H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/TrafficSound/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspectiv

RuntimeError: Dataset 'D://TrafficSound/data.yaml' error  Dataset 'D://TrafficSound/data.yaml' images not found, missing path 'D:\TrafficSound\dataset\images\val'
Note dataset download directory is 'D:\taffic sign\datasets'. You can update this in 'C:\Users\aufaw_nq8olti\AppData\Roaming\Ultralytics\settings.json'

In [ ]:
# import cv2
# import numpy as np
# import os
# import random
# import yaml
# from ultralytics import YOLO

# # --- 1. ตั้งค่า Path (ปรับให้ตรงกับเครื่องคุณ) ---
# raw_path = r'D:/TrafficSound/raw_images'  # ที่เก็บรูป stop.jpg, warning.jpg
# base_path = r'D:/TrafficSound/dataset'
# img_train_path = os.path.join(base_path, 'images/train')
# lbl_train_path = os.path.join(base_path, 'labels/train')

# # สร้างโฟลเดอร์โครงสร้าง YOLO
# os.makedirs(img_train_path, exist_ok=True)
# os.makedirs(lbl_train_path, exist_ok=True)

# # --- 2. ฟังก์ชันปั่นข้อมูล (Augmentation) เพื่อสอนเรื่อง "รูปทรง" ---
# def generate_data(img_name, class_id, num_variants=150):
#     full_path = os.path.join(raw_path, img_name)
#     img = cv2.imread(full_path)
#     if img is None:
#         print(f"Warning: ไม่พบไฟล์ {img_name} ใน {raw_path}")
#         return

#     name_only = os.path.splitext(img_name)[0]
    
#     for i in range(num_variants):
#         # สุ่มหมุนภาพ (Rotate) เพื่อให้ AI จำทรงแปดเหลี่ยม/ข้าวหลามตัดได้ทุกมุม
#         angle = random.randint(-20, 20)
#         h, w = img.shape[:2]
#         M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
#         variant = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REPLICATE)

#         # สุ่มปรับแสง (Brightness/Contrast)
#         contrast = random.uniform(0.7, 1.3)
#         brightness = random.randint(-30, 30)
#         variant = cv2.convertScaleAbs(variant, alpha=contrast, beta=brightness)

#         # บันทึกรูปภาพ
#         save_name = f"{name_only}_{i}.jpg"
#         cv2.imwrite(os.path.join(img_train_path, save_name), variant)

#         # สร้าง Label (0.5 0.5 1.0 1.0 คือ ป้ายอยู่เต็มกรอบ)
#         with open(os.path.join(lbl_train_path, f"{name_only}_{i}.txt"), 'w') as f:
#             f.write(f"{class_id} 0.5 0.5 1.0 1.0")

# # --- 3. การจัดการ "วัตถุที่ไม่ใช่ป้าย" (Background/Negative Samples) ---
# def add_background_images(bg_folder_name):
#     # ถ้าคุณมีรูปพวก 'ไฟท้ายรถ' หรือ 'ถังขยะสีแดง' ให้ใส่ไว้ในโฟลเดอร์นี้
#     bg_full_path = os.path.join(raw_path, bg_folder_name)
#     if not os.path.exists(bg_full_path): return

#     for filename in os.listdir(bg_full_path):
#         if filename.endswith(('.jpg', '.png')):
#             img = cv2.imread(os.path.join(bg_full_path, filename))
#             # บันทึกรูปเข้าชุดเทรน แต่ "ไม่ต้องสร้างไฟล์ .txt" 
#             # YOLO จะเรียนรู้ว่าถ้าไม่มีไฟล์ .txt คือพื้นที่ว่างที่ห้าม Detect มั่ว
#             cv2.imwrite(os.path.join(img_train_path, f"bg_{filename}"), img)

# # --- 4. รันกระบวนการเตรียมข้อมูล ---
# print("กำลังสร้าง Dataset...")
# generate_data('stop.jpg', 0)    # คลาส 0
# generate_data('warning.jpg', 1) # คลาส 1
# # add_background_images('others') # (Optional) ใส่รูปที่ไม่ใช่ป้ายเพื่อลด Error

# # --- 5. สร้าง data.yaml ---
# config = {
#     'path': base_path,
#     'train': 'images/train',
#     'val': 'images/train', # ใช้ร่วมกันเพื่อความรวดเร็วในขั้นตอนนี้
#     'names': {0: 'STOP', 1: 'WARNING'}
# }
# with open(os.path.join(base_path, 'data.yaml'), 'w') as f:
#     yaml.dump(config, f)

# # --- 6. เทรนและส่งออกโมเดล ---
# print("กำลังเทรน YOLOv8...")
# model = YOLO('yolov8n.pt') 
# model.train(data=os.path.join(base_path, 'data.yaml'), epochs=50, imgsz=640, device='cpu')

# # Export เป็น ONNX ไปใช้ใน Visual Studio
# path_onnx = model.export(format='onnx', opset=11)
# print(f"สำเร็จ! โมเดลอยู่ที่: {path_onnx}")

กำลังสร้าง Dataset...


FileNotFoundError: [Errno 2] No such file or directory: 'D:/TrafficSound/raw_images\\stop.jpg'

In [59]:
print(dataset.class_to_idx)

{'not_sign': 0, 'red_sign': 1, 'yellow_sign': 2}
